# 43) Koşullu Olasılık Nedir?
Koşullu Olasılık, bir olayın (A), **başka bir olayın (B) gerçekleştiği bilindiğinde** gerçekleşme olasılığını ifade eder. Gösterimi $P(A|B)$'dir ve "B verildiğinde A'nın olasılığı" diye okunur.

Bu, normal (koşulsuz) olasılıktan farklıdır: $P(A)$ sadece A'nın genel 
olasılığını sorar, $P(A|B)$ ise "elimizde B hakkında bir bilgi varken, bu 
bilgi A'nın olasılığını nasıl değiştirir" sorusunu sorar. Yeni bilgi, 
olasılığı **günceller.**

## Formül
$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

Burada $P(A \cap B)$, A ve B'nin **ikisinin birden** gerçekleşme olasılığı 
(kesişim). Formül aslında şunu söylüyor: "B'nin gerçekleştiği evrende, A 
ile B'nin birlikte gerçekleştiği kısmın oranı nedir?"

## Somut Örnek Kart Destesi
Standart bir 52 kartlık deste düşünelim.
- $P(\text{Kırmızı})$ = 26/52 = 0.5 (koşulsuz olasılık kartın rengi 
  hakkında hiçbir ek bilgi yok)
- Şimdi birisi bize "çekilen kart bir **resimli kart** (Vale, Kız, Papaz)" 
  dediğini varsayalım. Bu bilgiyle, kartın kırmızı olma olasılığı nedir?
$$P(\text{Kırmızı} \mid \text{Resimli}) = \frac{P(\text{Kırmızı ve Resimli})}{P(\text{Resimli})} = \frac{6/52}{12/52} = 0.5$$

İlginç bir sonuç: bu örnekte koşullu olasılık (0.5) ile koşulsuz olasılık (0.5) **aynı** çıktı! Bu, "resimli olması" bilgisinin, kartın renginin olasılığını hiç değiştirmediği anlamına gelir.

## Bağımsızlık ile Bağlantı (Kritik Nokta)
Eğer $P(A|B) = P(A)$ ise (yukarıdaki örnekteki gibi), A ve B olayları **bağımsızdır** yani B hakkında bilgi sahibi olmak, A'nın olasılığını hiç değiştirmez. Bu, Bernoulli/Binom'un "denemelerin bağımsız olmalı" koşulunun **resmi/matematiksel tanımıdır.**

Eğer $P(A|B) \ne P(A)$ ise, A ve B **bağımlıdır**. B hakkındaki bilgi, A'nın olasılığını değiştirir (artırır veya azaltır). İşte bu durumda koşullu olasılık gerçekten "yeni bilgi" katmış olur, ki bir sonraki konumuz olan **Bayes Teoremi**, tam olarak bu bağımlı durumları kullanarak tahminleri güncellemeye dayanır.

## Neden Önemli?
- İş dünyasında sürekli koşullu sorular sorarız: "Bir müşteri geçen ay 
  alışveriş yaptıysa, bu ay da yapma olasılığı nedir?", "Bir kullanıcı 
  reklamı gördüyse, tıklama olasılığı nedir?"
- A/B testlerinde ve pazarlama segmentasyonunda temel bir düşünme biçimi

In [1]:
import numpy as np
import pandas as pd

# --- Senaryo: 1000 kullanıcılık bir pazarlama verisi simüle edelim ---
# Reklamı görme ve satın alma durumlarını BİRLİKTE (gerçekçi bir ilişkiyle) üretelim
np.random.seed(42)

n = 1000
reklam_gordu = np.random.choice([1, 0], size=n, p=[0.4, 0.6])  # %40'ı reklamı gördü

# Satın alma olasılığı, reklamı görüp görmemesine göre FARKLI olsun (bağımlılık kuralım)
satin_alma = np.array([
    np.random.choice([1, 0], p=[0.35, 0.65]) if gordu == 1   # reklamı görenler %35 satın alıyor
    else np.random.choice([1, 0], p=[0.10, 0.90])            # görmeyenler sadece %10 satın alıyor
    for gordu in reklam_gordu
])

veri = pd.DataFrame({'reklam_gordu': reklam_gordu, 'satin_aldi': satin_alma})

# --- Koşulsuz Olasılıklar ---
p_satin_alma = veri['satin_aldi'].mean()
p_reklam = veri['reklam_gordu'].mean()
print(f"P(Satın Alma) = {p_satin_alma:.4f}")
print(f"P(Reklam Gördü) = {p_reklam:.4f}")

# --- Kesişim: Hem reklamı gördü HEM satın aldı ---
p_kesisim = ((veri['reklam_gordu']==1) & (veri['satin_aldi']==1)).mean()
print(f"P(Reklam VE Satın Alma) = {p_kesisim:.4f}")

# --- Koşullu Olasılık: Reklamı görenler arasında satın alma oranı ---
p_satinalma_verildi_reklam = p_kesisim / p_reklam
print(f"\nP(Satın Alma | Reklam Gördü) = {p_satinalma_verildi_reklam:.4f}")

# Doğrulama: Pandas ile direkt filtreleyerek de aynı sonucu bulalım
dogrulama = veri[veri['reklam_gordu']==1]['satin_aldi'].mean()
print(f"Doğrulama (direkt filtreleme): {dogrulama:.4f}")

# --- Karşılaştırma: Reklamı görmeyenlerde satın alma oranı ---
p_satinalma_verildi_reklamsiz = veri[veri['reklam_gordu']==0]['satin_aldi'].mean()
print(f"\nP(Satın Alma | Reklam Görmedi) = {p_satinalma_verildi_reklamsiz:.4f}")

P(Satın Alma) = 0.2000
P(Reklam Gördü) = 0.4210
P(Reklam VE Satın Alma) = 0.1410

P(Satın Alma | Reklam Gördü) = 0.3349
Doğrulama (direkt filtreleme): 0.3349

P(Satın Alma | Reklam Görmedi) = 0.1019


### Sonuç
Genel satın alma oranı %20 iken, reklamı görenler arasında satın alma oranı %33.49'a çıkıyor. Bu da reklamın satın alma olasılığını yaklaşık 1.5 katına çıkardığını gösteriyor.